# Imma Hungry (CIS 1921 Project)

Harry Li, Nabo Yu

We want to build an optimizer for the [Share Food Program](https://www.sharefoodprogram.org/philly-food-rescue).

In brief, the problem is as follows. We have a network of volunteer drivers to pick up perishable food from grocers, restaurants, and caterers, and they deliver them to affordable housing communities and nonprofits.

### Data Source

We use real world data from [Datasets - OpenDataPhilly](https://opendataphilly.org/datasets/?category=food), including [OpenMaps - OpenDataPhilly](https://opendataphilly.org/datasets/openmaps/) and [Free Food & Meal Sites - OpenDataPhilly](https://opendataphilly.org/datasets/free-meals/), and Google Maps API, including the [Compute Route Matrix](https://developers.google.com/maps/documentation/routes/compute-route-matrix-over) API.

- The OpenMaps dataset provides the geographic information of Philly.
- The Free Food & Meal Sites dataset provides the food donations information in Philly.
- The Compute Route Matrix API provides the travel route matrix (i.e., generalized distance matrix taking route and traffic into consideration) between the locations.

### Formulation

Let there be $X$ volunteer drivers with carrying capacity $c$ scattered around Philadelphia, $Y$ active food donors with $s$ hours of spoilage windows, and $Z$ drop-off locations with pantry capacity $p$. We will optimize the driver-donation assignments and the route for each assignment by minimizing *food waste* and maximizing *nutrient availability* in the process.

### Evaluation

We evaluate the system by running simulations of sampled location subsets in the OpenDataPhilly dataset.

1. With small-scale driver and location samples to verify our pipeline works.
2. With larger-scale driver and location samples to compare our solver with
   - a baseline greedy strategy, and
   - some heuristic strategy
   
   to see if our solver does indeed improve upon naive approaches.

## Data Source

In [ ]:
import polars as pl

print("OK!")

In [ ]:
def fetch_data(remote: str, local: str):
    import os

    if os.path.exists(local):
        return
    os.makedirs(os.path.dirname(local), exist_ok=True)
    import urllib.request

    urllib.request.urlretrieve(remote, local)

There is no public information on food donors. We use the information of restaurants instead, assuming they are willing to donate food.

In [ ]:
# https://opendataphilly.org/datasets/licenses-and-inspections-business-licenses/
fetch_data(
    "https://phl.carto.com/api/v2/sql?q=SELECT+*,+ST_Y(the_geom)+AS+lat,+ST_X(the_geom)+AS+lng+FROM+business_licenses&filename=business_licenses&format=csv&skipfields=cartodb_id",
    "data/business_licenses.csv",
)

donor_locations = pl.read_csv("data/business_licenses.csv", infer_schema_length=65536)

donor_locations = (
    donor_locations.filter(
        pl.col("licensestatus") == "Active",
        pl.col("licensetype").is_in(
            [
                "Food Preparing and Serving",
                "Food Preparing and Serving (30+)",
                "Food Establishment, Retail Permanent",
            ]
        ),
    )
    .select(
        pl.col("business_name"),
        pl.col("address"),
        pl.col("lat"),
        pl.col("lng"),
    )
    .drop_nulls(subset=["lat", "lng"])
)

donor_locations = donor_locations.sample(50, seed=1921)
donor_locations

We use the information of existing share food program sites.

In [ ]:
# https://opendataphilly.org/datasets/free-meals/
# spatialRefId=4326 gives WGS84 lat/lng directly in x (lng) and y (lat) columns
fetch_data(
    "https://hub.arcgis.com/api/v3/datasets/5825a32bb8844bb097f7a16d4fbf4f23_0/downloads/data?format=csv&spatialRefId=4326&where=1%3D1",
    "data/free_meal_sites.csv",
)

dropoff_locations = pl.read_csv("data/free_meal_sites.csv")

dropoff_locations = (
    dropoff_locations.filter(pl.col("category").str.contains("SHARE FOOD PROGRAM"))
    .select(
        pl.col("site_name"),
        pl.col("address"),
        pl.col("x").alias("lng"),
        pl.col("y").alias("lat"),
    )
    .drop_nulls(subset=["lat", "lng"])
)

dropoff_locations = dropoff_locations.sample(14, seed=1921)
dropoff_locations

We use the [OSRM Table API](http://router.project-osrm.org) (free, no key, OpenStreetMap-based) to compute driving distances and durations for all (donor + drop-off) → drop-off pairs in a single request.

Before querying, we filter pairs by **crow-flies (Haversine) distance**: pairs exceeding `MAX_CROW_FLIES_KM` are excluded from the output and treated as infeasible by the optimizer. This doesn't reduce OSRM calls (the table service always computes the full matrix), but it shrinks the optimizer's search space. If you later switch to the Google Maps Route Matrix API, the same filter will also skip billable elements.

**On duration and time-of-day**: OSRM uses static average speeds (no live traffic). For a planning/simulation context this is fine. If you need traffic-aware durations, switch to the Google Maps Routes API — the $200/month free credit covers ~10,000 route elements and should be sufficient for the project.

In [ ]:
import os
import numpy as np

# Set to "osrm" for free routing (no API key, uses OpenStreetMap data, no live traffic)
# Set to "google" for traffic-aware routing (requires GOOGLE_MAPS_API_KEY env var or ~/.gcp_api_key)
ROUTING_BACKEND = "osrm"


def _load_google_api_key():
    key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    if not key:
        keyfile = os.path.expanduser("~/.gcp_api_key")
        if os.path.exists(keyfile):
            with open(keyfile) as f:
                key = f.read().strip()
    return key


GOOGLE_API_KEY = _load_google_api_key()

# Pairs with crow-flies distance beyond this are treated as infeasible by the optimizer.
# ~10 km keeps routes within a reasonable volunteer driving range in Philly (~6 miles).
# For the Google backend this also skips those elements from the API request (saves cost).
MAX_CROW_FLIES_KM = 10.0

# Sentinel used in distance/duration matrices for non-viable or unreachable pairs.
# Large enough that no optimizer will select these edges; small enough to avoid int32 overflow when summed.
INFEASIBLE = 10**7  # ~10,000 km / ~115 days — clearly unreachable
# Turns out this still causes overflow; we'll deal with it later on


def haversine_matrix(lat1, lng1, lat2, lng2):
    """Returns (m, n) matrix of crow-flies distances in km."""
    lat1 = np.radians(np.asarray(lat1, dtype=float))[:, None]
    lng1 = np.radians(np.asarray(lng1, dtype=float))[:, None]
    lat2 = np.radians(np.asarray(lat2, dtype=float))[None, :]
    lng2 = np.radians(np.asarray(lng2, dtype=float))[None, :]
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))


def _fetch_osrm(n_donors, n_dropoffs, all_lat, all_lng):
    """Returns (durations_s, distances_m) as (n_total, n_dropoffs) float arrays. NaN = unreachable."""
    import json, urllib.request

    n_total = n_donors + n_dropoffs
    coords = ";".join(f"{lng},{lat}" for lat, lng in zip(all_lat, all_lng))
    src_idx = ";".join(str(i) for i in range(n_total))
    dst_idx = ";".join(str(i) for i in range(n_total))
    url = (
        f"http://router.project-osrm.org/table/v1/driving/{coords}"
        f"?sources={src_idx}&destinations={dst_idx}"
        f"&annotations=duration,distance&generate_hints=false"
    )
    with urllib.request.urlopen(url, timeout=60) as resp:
        data = json.loads(resp.read())
    return (
        np.array(data["durations"], dtype=float),  # (n_total, n_dropoffs)
        np.array(data["distances"], dtype=float),  # (n_total, n_dropoffs)
    )


def _fetch_google(viable, n_donors, n_dropoffs, all_addresses):
    """Returns (durations_s, distances_m) as (n_total, n_dropoffs) float arrays. NaN = unreachable."""
    from google.api_core.client_options import ClientOptions
    import google.maps.routing_v2 as routes_api

    client = routes_api.RoutesClient(
        client_options=ClientOptions(api_key=GOOGLE_API_KEY)
    )
    n_total = n_donors + n_dropoffs

    all_origins = [
        routes_api.RouteMatrixOrigin(
            waypoint=routes_api.Waypoint(address=all_addresses[i])
        )
        for i in range(n_total)
    ]
    all_destinations = [
        routes_api.RouteMatrixDestination(waypoint=routes_api.Waypoint(address=all_addresses[i]))
        for i in range(n_total)
    ]

    # Skip origins with no viable destinations, saves billable API elements
    viable_origins = [i for i in range(n_total) if viable[i].any()]
    if skipped := n_total - len(viable_origins):
        print(f"  Skipping {skipped} origins with no viable destinations")

    durations = np.full((n_total, n_total), np.nan)
    distances = np.full((n_total, n_total), np.nan)

    BATCH_SIZE = 50 - n_dropoffs  # origins + destinations <= 50 per request
    for start in range(0, len(viable_origins), BATCH_SIZE):
        batch_indices = viable_origins[start : start + BATCH_SIZE]
        response = client.compute_route_matrix(
            routes_api.ComputeRouteMatrixRequest(
                origins=[all_origins[i] for i in batch_indices],
                destinations=all_destinations,
            ),
            metadata=[
                (
                    "x-goog-fieldmask",
                    "originIndex,destinationIndex,duration,distanceMeters,status",
                )
            ],
        )
        for el in response:
            i, j = batch_indices[el.origin_index], el.destination_index
            distances[i, j] = el.distance_meters
            durations[i, j] = el.duration.seconds

    return durations, distances


def fetch_matrix():
    n_donors = len(donor_locations)
    n_dropoffs = len(dropoff_locations)
    CACHE_PATH = f"data/route_matrix_{ROUTING_BACKEND}_{n_donors}d_{n_dropoffs}p.npz"

    if os.path.exists(CACHE_PATH):
        data = np.load(CACHE_PATH)
        return {
            "locations": data["locations"].tolist(),
            "n_donors": int(data["n_donors"]),
            "distance": data["distance"],
            "duration": data["duration"],
        }

    all_lat = donor_locations["lat"].to_list() + dropoff_locations["lat"].to_list()
    all_lng = donor_locations["lng"].to_list() + dropoff_locations["lng"].to_list()
    locations = (
        donor_locations["business_name"].to_list()
        + dropoff_locations["site_name"].to_list()
    )

    # Crow-flies filter: only road-distance/duration for nearby pairs
    crow_km = haversine_matrix(all_lat, all_lng, all_lat, all_lng)
    viable = crow_km <= MAX_CROW_FLIES_KM
    n_total = n_donors + n_dropoffs
    print(f"Crow-flies filter ({ROUTING_BACKEND}): {viable.sum()} / {n_total * n_total} pairs viable")

    if ROUTING_BACKEND == "osrm":
        raw_dur, raw_dist = _fetch_osrm(n_donors, n_dropoffs, all_lat, all_lng)
    elif ROUTING_BACKEND == "google":
        if not GOOGLE_API_KEY:
            raise ValueError(
                "No API key found — set GOOGLE_MAPS_API_KEY env var or write it to ~/.gcp_api_key"
            )
        all_addresses = [
            f"{a}, Philadelphia, PA" for a in donor_locations["address"].to_list()
        ] + [f"{a}, Philadelphia, PA" for a in dropoff_locations["address"].to_list()]
        raw_dur, raw_dist = _fetch_google(viable, n_donors, n_dropoffs, all_addresses)
    else:
        raise ValueError(
            f"Unknown ROUTING_BACKEND {ROUTING_BACKEND!r} — use 'osrm' or 'google'"
        )

    # Merge viable mask with reachability; fill everything else with INFEASIBLE sentinel
    ok = viable & ~np.isnan(raw_dist) & ~np.isnan(raw_dur)
    distance = np.where(ok, raw_dist, INFEASIBLE).astype(np.int32)
    duration = np.where(ok, raw_dur, INFEASIBLE).astype(np.int32)

    os.makedirs("data", exist_ok=True)
    np.savez_compressed(
        CACHE_PATH,
        locations=np.array(locations),
        n_donors=np.array([n_donors]),
        distance=distance,
        duration=duration,
    )
    return {
        "locations": locations,
        "n_donors": n_donors,
        "distance": distance,
        "duration": duration,
    }


# route_matrix["locations"][i]  - name of location i
# route_matrix["n_donors"]      - locations[:n_donors] are donors; the rest are dropoffs
# route_matrix["distance"][i,j] - road distance (meters) from location i to dropoff (n_donors+j)
# route_matrix["duration"][i,j] - driving time (seconds); INFEASIBLE sentinel where not viable
route_matrix = fetch_matrix()

nd, n = route_matrix["n_donors"], len(route_matrix["locations"])
viable_count = (route_matrix["distance"] < INFEASIBLE).sum()
print(f"{nd} donors + {n - nd} dropoffs | {viable_count} viable pairs")
route_matrix["distance"]

## Optimization Formulation

In [ ]:
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [ ]:
nd = route_matrix["n_donors"]
n = len(route_matrix["locations"])
num_dropoffs = n - nd
num_vehicles = 5
depot_index = 0  # Assuming driver starts at location 0

In [ ]:
np.random.seed(1921)

In [ ]:
# Weight of food available: Donors 1 to nd (Depot gets 0)
donor_total_weight = np.random.randint(10, 60, size=nd)
donor_total_weight[0] = 0  # Depot has no food

In [ ]:
# Nutrients [Fiber, Protein, Carbs, Micro] split
proportions = np.random.dirichlet(np.ones(4), size=nd)
donor_nutrients = np.floor(donor_total_weight[:, None] * proportions).astype(int)

In [ ]:
# Time windows: Food spoils 2 hours (7200s) after it's ready.
# Window: 0 to 4 hours (14400s) for opening.
donor_time_open = np.random.randint(0, 14400, size=nd)
donor_time_open[0] = 0  # Depot always open
donor_time_close = donor_time_open + 7200
donor_time_close[0] = 28800  # Depot open all day (8 hours)

In [ ]:
dropoff_capacity = np.random.randint(50, 200, size=num_dropoffs)

In [ ]:
# Relaxed nutrient balancing cap. Keep as a tuning parameter.
# 0.40 as a hard cap is often too restrictive with random synthetic donor nutrient mixes.
NUTRIENT_CAP_FRACTION = 0.85
dropoff_nutrient_max = np.floor(dropoff_capacity * NUTRIENT_CAP_FRACTION).astype(int)

In [ ]:
# Pantries open all day
dropoff_time_open = np.zeros(num_dropoffs, dtype=int)
dropoff_time_close = np.full(num_dropoffs, 28800, dtype=int)

In [ ]:
time_windows = [
    (int(o), int(c))
    for o, c in zip(
        np.concatenate([donor_time_open, dropoff_time_open]), np.concatenate([donor_time_close, dropoff_time_close])
    )
]

In [ ]:
# Virtual Node Setup
num_pantries = n - nd
total_virtual_nodes = 2 * nd - 1  # Depot + Donors + 1 Virtual Pantry for every Donor

# Assign each donor to its nearest feasible pantry (if any).
donor_to_pantry = np.full(nd, -1, dtype=int)
for donor in range(1, nd):
    donor_to_pantry_distances = route_matrix["distance"][donor, nd:n]
    feasible_mask = donor_to_pantry_distances < INFEASIBLE
    if feasible_mask.any():
        best_local_idx = int(np.argmin(np.where(feasible_mask, donor_to_pantry_distances, np.inf)))
        donor_to_pantry[donor] = best_local_idx


def get_real_node(v_node):
    # Maps our virtual graph back to real distance matrix
    if v_node < nd:
        return v_node

    donor_idx = v_node - nd + 1
    pantry_local_idx = int(donor_to_pantry[donor_idx])
    if pantry_local_idx < 0:
        # Fallback: if a donor has no feasible pantry, map to first pantry.
        # That pair will still tend to be dropped because route cost is infeasible-sized.
        return nd
    return nd + pantry_local_idx

In [ ]:
vehicle_capacities = [200] * num_vehicles

In [ ]:
manager = pywrapcp.RoutingIndexManager(total_virtual_nodes, num_vehicles, depot_index)
routing = pywrapcp.RoutingModel(manager)

In [ ]:
# Create a list to hold strong references to our callbacks
# If Python garbage collects these, C++ segfaults or overflows
callbacks_store = []

In [ ]:
def distance_callback(from_index, to_index):
    try:
        from_real = get_real_node(manager.IndexToNode(from_index))
        to_real = get_real_node(manager.IndexToNode(to_index))
        val = float(route_matrix["distance"][from_real][to_real])
        if np.isnan(val) or np.isinf(val) or val > 999999:
            return 999999
        return int(val)
    except Exception:
        return 999999


callbacks_store.append(distance_callback)
transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)  # Minimize distance

In [ ]:
def time_callback(from_index, to_index):
    try:
        from_real = get_real_node(manager.IndexToNode(from_index))
        to_real = get_real_node(manager.IndexToNode(to_index))
        val = float(route_matrix["duration"][from_real][to_real])
        if np.isnan(val) or np.isinf(val) or val > 999999:
            return 999999
        service_time = 600 if from_real != depot_index else 0
        return int(val + service_time)
    except Exception:
        return 999999


callbacks_store.append(time_callback)
time_callback_index = routing.RegisterTransitCallback(time_callback)
# Allow 8 hours total max, allow waiting if arriving early
routing.AddDimension(
    time_callback_index,
    28800,  # allow waiting time
    28800,  # maximum time per vehicle
    False,  # Don't force start to 0
    "Time",
)
time_dimension = routing.GetDimensionOrDie("Time")

In [ ]:
# Apply the time windows (Spoilage limits) to the Virtual Nodes
for v_node in range(total_virtual_nodes):
    # Skip the depot; we must handle it per-vehicle below.
    if v_node == 0:
        continue

    index = manager.NodeToIndex(v_node)
    if index == -1:
        continue

    # Figure out which real-world location this virtual node represents
    real_node = get_real_node(v_node)
    t_open, t_close = time_windows[real_node]

    # Safely apply the range
    time_var = time_dimension.CumulVar(index)
    if time_var is not None:
        time_var.SetRange(int(t_open), int(t_close))

# Explicitly apply the Depot time windows to all 5 vehicles
depot_open, depot_close = time_windows[depot_index]

for vehicle_id in range(num_vehicles):
    start_index = routing.Start(vehicle_id)
    end_index = routing.End(vehicle_id)

    start_var = time_dimension.CumulVar(start_index)
    if start_var is not None:
        start_var.SetRange(int(depot_open), int(depot_close))

    end_var = time_dimension.CumulVar(end_index)
    if end_var is not None:
        end_var.SetRange(int(depot_open), int(depot_close))

In [ ]:
# Vehicle Capacity (Hard Constraint)
def demand_callback(from_index):
    try:
        v_node = manager.IndexToNode(from_index)
        if v_node == 0:
            return 0
        if v_node < nd:
            return int(donor_total_weight[v_node])
        else:
            # This completely empties the exact amount picked up
            matched_donor = v_node - nd + 1
            return -int(donor_total_weight[matched_donor])
    except Exception:
        return 0


callbacks_store.append(demand_callback)
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  # null capacity slack
    vehicle_capacities,  # vehicle maximum capacities
    True,  # start cumul to zero
    "Weight",
)

In [ ]:
# Enforce Pickup -> Delivery
for donor_v_node in range(1, nd):
    dropoff_v_node = nd + donor_v_node - 1

    pickup_idx = manager.NodeToIndex(donor_v_node)
    delivery_idx = manager.NodeToIndex(dropoff_v_node)

    if pickup_idx == -1 or delivery_idx == -1:
        continue

    # Canonical optional pickup-delivery pattern:
    # add separate disjunctions and tie active statuses together.
    per_node_drop_penalty = 1_000_000
    routing.AddDisjunction([pickup_idx], per_node_drop_penalty)
    routing.AddDisjunction([delivery_idx], per_node_drop_penalty)

    routing.AddPickupAndDelivery(pickup_idx, delivery_idx)
    routing.solver().Add(routing.VehicleVar(pickup_idx) == routing.VehicleVar(delivery_idx))
    routing.solver().Add(routing.ActiveVar(pickup_idx) == routing.ActiveVar(delivery_idx))
    # Enforce that pickup happens before delivery
    routing.solver().Add(time_dimension.CumulVar(pickup_idx) <= time_dimension.CumulVar(delivery_idx))

SPOILAGE_LIMIT = 7200  # 2 hours in seconds
routing.solver().Add(time_dimension.CumulVar(delivery_idx) <= time_dimension.CumulVar(pickup_idx) + SPOILAGE_LIMIT)

In [ ]:
nutrient_names = ["Fiber", "Protein", "Carbs", "Micro"]
solver = routing.solver()

# Group donors by the pantry they are assigned to (nearest feasible pantry).
donors_for_pantry = {p: [] for p in range(num_pantries)}
for donor in range(1, nd):
    pantry_local_idx = int(donor_to_pantry[donor])
    if pantry_local_idx >= 0:
        donors_for_pantry[pantry_local_idx].append(donor)

for real_pantry, donors in donors_for_pantry.items():
    if not donors:
        continue
    for k, nutrient in enumerate(nutrient_names):
        nutrient_sum = solver.Sum(
            [routing.ActiveVar(manager.NodeToIndex(d)) * int(donor_nutrients[d][k]) for d in donors]
        )
        solver.Add(nutrient_sum <= int(dropoff_nutrient_max[real_pantry]))
    weight_sum = solver.Sum(
        [routing.ActiveVar(manager.NodeToIndex(d)) * int(donor_total_weight[d]) for d in donors]
    )
    solver.Add(weight_sum <= int(dropoff_capacity[real_pantry]))

In [ ]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
# Insertion-based first solution is usually stronger for pickup-delivery with optional nodes.
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_parameters.time_limit.seconds = 30

In [ ]:
print("Solving routes with pickups and dropoffs...")
solution = routing.SolveWithParameters(search_parameters)

In [ ]:
status_names = {
    0: "ROUTING_NOT_SOLVED",
    1: "ROUTING_SUCCESS",
    2: "ROUTING_PARTIAL_SUCCESS_LOCAL_OPTIMUM_NOT_REACHED",
    3: "ROUTING_FAIL",
    4: "ROUTING_FAIL_TIMEOUT",
    5: "ROUTING_INVALID",
    6: "ROUTING_INFEASIBLE",
}
print(f"Status: {status_names.get(routing.status(), routing.status())}")

if solution:
    weight_dim = routing.GetDimensionOrDie("Weight")
    total_distance = 0
    total_food_rescued = 0

    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        plan_output = f"Route for driver {vehicle_id}:\n"
        route_distance = 0

        while not routing.IsEnd(index):
            v_node = manager.IndexToNode(index)
            real_node = get_real_node(v_node)
            loc_name = route_matrix["locations"][real_node]

            # Get current time and truck load
            time_var = solution.Min(time_dimension.CumulVar(index))
            load_var = solution.Min(weight_dim.CumulVar(index))

            if v_node == 0:
                node_type = "[Depot]"
            elif v_node < nd:
                weight = int(donor_total_weight[v_node])
                node_type = f"[Pickup +{weight}lbs]"
                total_food_rescued += weight
            else:
                matched_donor = v_node - nd + 1
                weight = int(donor_total_weight[matched_donor])
                node_type = f"[Dropoff -{weight}lbs]"

            plan_output += f" {node_type} {loc_name} (Truck Load: {load_var}lbs) -> \n"

            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        plan_output += f" [Depot] Return (Distance: {route_distance}m)\n"
        print(plan_output)
        total_distance += route_distance

    print(f"\nTotal Distance: {total_distance}m")
    print(f"Total Food Rescued and Delivered: {total_food_rescued}lbs")
else:
    print("No solution found!")

In [ ]:
# Diagnostics: verify solver isn't collapsing to all-dropped solution
if solution:
    served_pairs = 0
    dropped_pairs = 0
    for donor_v_node in range(1, nd):
        pickup_idx = manager.NodeToIndex(donor_v_node)
        delivery_idx = manager.NodeToIndex(nd + donor_v_node - 1)
        if pickup_idx == -1 or delivery_idx == -1:
            continue
        active_pickup = solution.Value(routing.ActiveVar(pickup_idx))
        active_delivery = solution.Value(routing.ActiveVar(delivery_idx))
        if active_pickup == 1 and active_delivery == 1:
            served_pairs += 1
        elif active_pickup == 0 and active_delivery == 0:
            dropped_pairs += 1

    print(f"Served pickup-delivery pairs: {served_pairs} / {nd - 1}")
    print(f"Dropped pickup-delivery pairs: {dropped_pairs} / {nd - 1}")
    print(f"Objective value (distance + drop penalties): {solution.ObjectiveValue()}")
else:
    print("No solution found")